# Phase 1A Deliverable 1 and 2 Validation

This notebook is a human-readable acceptance check for Phase 1A. It does not replace pytest coverage; it gives visible spot checks for the two deliverables:

- Deliverable 1: `backtest/bt_parquet_feed.py`
- Deliverable 2: `backtest/execution_model.py` and `backtest/slippage.py`

Run the notebook top to bottom. A failed deliverable check raises an assertion error in the cell where the mismatch is found.

## Setup

In [1]:
from pathlib import Path
import tempfile

import backtrader as bt
import numpy as np
import pandas as pd
from IPython.display import display

from backtest.bt_parquet_feed import ParquetFeed
from backtest.execution_model import AlphaBroker, configure_cerebro

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

PARQUET_PATH = Path("data/raw/btcusdt_1h.parquet")
COMMISSION_RATE = 0.0007

assert PARQUET_PATH.exists(), f"Missing canonical parquet: {PARQUET_PATH}"
print("Validation target:", PARQUET_PATH)
print("Expected commission rate:", COMMISSION_RATE)


Validation target: data/raw/btcusdt_1h.parquet
Expected commission rate: 0.0007


## Part A - Deliverable 1: ParquetFeed

The feed passes this section if:

- Backtrader sees the same row count as pandas.
- Backtrader datetimes align one-for-one with `open_time_utc` after stripping UTC tzinfo for Backtrader.
- OHLCV values match the direct pandas read.
- `fromdate` and `todate` filtering returns the same inclusive range as pandas.

### A1. Direct pandas read of the canonical parquet

In [2]:
raw_df = (
    pd.read_parquet(PARQUET_PATH)
    .sort_values("open_time_utc")
    .reset_index(drop=True)
)

required_cols = ["open_time_utc", "open", "high", "low", "close", "volume"]
missing_cols = set(required_cols) - set(raw_df.columns)
assert not missing_cols, f"Canonical parquet missing columns: {sorted(missing_cols)}"

print("Rows:", len(raw_df))
print("Start:", raw_df["open_time_utc"].min())
print("End:", raw_df["open_time_utc"].max())
display(raw_df.head(5)[required_cols])


Rows: 55105
Start: 2020-01-01 00:00:00+00:00
End: 2026-04-16 07:00:00+00:00


,open_time_utc,open,high,low,close,volume
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 03:00:00+00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 04:00:00+00:00,7225.00,7230.00,7215.03,7217.27,467.812578


### A2. Full feed alignment against pandas

In [3]:
class CollectFeedStrategy(bt.Strategy):
    def __init__(self):
        self.rows = []

    def next(self):
        self.rows.append({
            "dt": self.data.datetime.datetime(0),
            "open": float(self.data.open[0]),
            "high": float(self.data.high[0]),
            "low": float(self.data.low[0]),
            "close": float(self.data.close[0]),
            "volume": float(self.data.volume[0]),
        })


def run_feed_collection(feed):
    cerebro = bt.Cerebro()
    cerebro.adddata(feed)
    cerebro.addstrategy(CollectFeedStrategy)
    return cerebro.run()[0]

feed = ParquetFeed.from_parquet(PARQUET_PATH)
collector = run_feed_collection(feed)
feed_df = pd.DataFrame(collector.rows)

expected_df = raw_df[required_cols].copy()
expected_dt_naive = expected_df["open_time_utc"].dt.tz_localize(None).reset_index(drop=True)

assert len(feed_df) == len(raw_df), f"Row count mismatch: feed={len(feed_df)}, pandas={len(raw_df)}"
pd.testing.assert_series_equal(feed_df["dt"].reset_index(drop=True), expected_dt_naive, check_names=False, check_dtype=False)

for col in ["open", "high", "low", "close", "volume"]:
    np.testing.assert_allclose(feed_df[col].to_numpy(), expected_df[col].to_numpy(), rtol=0, atol=1e-12)

comparison = pd.concat(
    [
        expected_df.head(5).rename(columns={"open_time_utc": "pandas_dt"}).reset_index(drop=True),
        feed_df.head(5).add_prefix("feed_").reset_index(drop=True),
    ],
    axis=1,
)

display(comparison)
print("PASS: Deliverable 1 full-feed row count, datetime alignment, and OHLCV values match pandas.")


,pandas_dt,open,high,low,close,volume,feed_dt,feed_open,feed_high,feed_low,feed_close,feed_volume
0,2020-01-01 00:00:00+00:00,7195.24,7196.25,7175.46,7177.02,511.814901,2020-01-01 00:00:00,7195.24,7196.25,7175.46,7177.02,511.814901
1,2020-01-01 01:00:00+00:00,7176.47,7230.00,7175.71,7216.27,883.052603,2020-01-01 01:00:00,7176.47,7230.00,7175.71,7216.27,883.052603
2,2020-01-01 02:00:00+00:00,7215.52,7244.87,7211.41,7242.85,655.156809,2020-01-01 02:00:00,7215.52,7244.87,7211.41,7242.85,655.156809
3,2020-01-01 03:00:00+00:00,7242.66,7245.00,7220.00,7225.01,783.724867,2020-01-01 03:00:00,7242.66,7245.00,7220.00,7225.01,783.724867
4,2020-01-01 04:00:00+00:00,7225.00,7230.00,7215.03,7217.27,467.812578,2020-01-01 04:00:00,7225.00,7230.00,7215.03,7217.27,467.812578


PASS: Deliverable 1 full-feed row count, datetime alignment, and OHLCV values match pandas.


### A2b. Extra Backtrader lines: quote volume and trade count

`ParquetFeed` exposes `quote_volume` and `trade_count` as custom Backtrader lines. This spot-check confirms those extra lines match the direct pandas read for the first five bars.

In [4]:
extra_cols = ["quote_volume", "trade_count"]
missing_extra_cols = set(extra_cols) - set(raw_df.columns)
assert not missing_extra_cols, f"Canonical parquet missing extra feed columns: {sorted(missing_extra_cols)}"


class CollectExtraLinesStrategy(bt.Strategy):
    def __init__(self):
        self.rows = []

    def next(self):
        if len(self.rows) < 5:
            self.rows.append({
                "dt": self.data.datetime.datetime(0),
                "quote_volume": float(self.data.quote_volume[0]),
                "trade_count": float(self.data.trade_count[0]),
            })


feed = ParquetFeed.from_parquet(PARQUET_PATH)
cerebro = bt.Cerebro()
cerebro.adddata(feed)
cerebro.addstrategy(CollectExtraLinesStrategy)
extra_strat = cerebro.run()[0]
extra_feed_df = pd.DataFrame(extra_strat.rows)

expected_extra_df = raw_df[["open_time_utc", "quote_volume", "trade_count"]].head(5).copy()
expected_extra_dt_naive = expected_extra_df["open_time_utc"].dt.tz_localize(None).reset_index(drop=True)

pd.testing.assert_series_equal(extra_feed_df["dt"].reset_index(drop=True), expected_extra_dt_naive, check_names=False, check_dtype=False)
np.testing.assert_allclose(extra_feed_df["quote_volume"].to_numpy(), expected_extra_df["quote_volume"].to_numpy(), rtol=0, atol=1e-12)
np.testing.assert_allclose(extra_feed_df["trade_count"].to_numpy(), expected_extra_df["trade_count"].astype(float).to_numpy(), rtol=0, atol=1e-12)

extra_comparison = pd.concat(
    [
        expected_extra_df.rename(columns={"open_time_utc": "pandas_dt"}).reset_index(drop=True),
        extra_feed_df.add_prefix("feed_").reset_index(drop=True),
    ],
    axis=1,
)

display(extra_comparison)
print("PASS: Deliverable 1 extra lines quote_volume and trade_count match pandas for the first five bars.")


,pandas_dt,quote_volume,trade_count,feed_dt,feed_quote_volume,feed_trade_count
0,2020-01-01 00:00:00+00:00,3.675857e+06,7640,2020-01-01 00:00:00,3.675857e+06,7640.0
1,2020-01-01 01:00:00+00:00,6.365953e+06,9033,2020-01-01 01:00:00,6.365953e+06,9033.0
2,2020-01-01 02:00:00+00:00,4.736719e+06,7466,2020-01-01 02:00:00,4.736719e+06,7466.0
3,2020-01-01 03:00:00+00:00,5.667367e+06,8337,2020-01-01 03:00:00,5.667367e+06,8337.0
4,2020-01-01 04:00:00+00:00,3.379094e+06,5896,2020-01-01 04:00:00,3.379094e+06,5896.0


PASS: Deliverable 1 extra lines quote_volume and trade_count match pandas for the first five bars.


### A3. `fromdate` / `todate` slicing

In [5]:
start_utc = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
end_utc = pd.Timestamp("2024-01-03 23:00:00", tz="UTC")

expected_slice = raw_df[
    (raw_df["open_time_utc"] >= start_utc) &
    (raw_df["open_time_utc"] <= end_utc)
].reset_index(drop=True)

assert len(expected_slice) > 0, "Expected the canonical parquet to contain the validation date range"
expected_first = expected_slice["open_time_utc"].iloc[0].tz_localize(None)
expected_last = expected_slice["open_time_utc"].iloc[-1].tz_localize(None)

range_cases = [
    (
        "naive datetime interpreted as UTC",
        start_utc.to_pydatetime().replace(tzinfo=None),
        end_utc.to_pydatetime().replace(tzinfo=None),
    ),
    (
        "UTC-aware datetime",
        start_utc.to_pydatetime(),
        end_utc.to_pydatetime(),
    ),
]

slice_summaries = []
case_frames = []
for label, from_arg, to_arg in range_cases:
    feed = ParquetFeed.from_parquet(PARQUET_PATH, fromdate=from_arg, todate=to_arg)
    collector = run_feed_collection(feed)
    slice_feed_df = pd.DataFrame(collector.rows)
    case_frames.append(slice_feed_df)

    assert len(slice_feed_df) == len(expected_slice), (
        f"{label} count mismatch: feed={len(slice_feed_df)}, pandas={len(expected_slice)}"
    )
    assert slice_feed_df["dt"].iloc[0] == expected_first
    assert slice_feed_df["dt"].iloc[-1] == expected_last

    for col in ["open", "high", "low", "close", "volume"]:
        np.testing.assert_allclose(slice_feed_df[col].to_numpy(), expected_slice[col].to_numpy(), rtol=0, atol=1e-12)

    slice_summaries.append({
        "case": label,
        "from_arg_type": type(from_arg).__name__,
        "from_arg_tzinfo": str(getattr(from_arg, "tzinfo", None)),
        "to_arg_tzinfo": str(getattr(to_arg, "tzinfo", None)),
        "count": len(slice_feed_df),
        "first_dt": slice_feed_df["dt"].iloc[0],
        "last_dt": slice_feed_df["dt"].iloc[-1],
    })

pd.testing.assert_frame_equal(case_frames[0], case_frames[1], check_dtype=False)

display(pd.DataFrame(slice_summaries))
print("Expected count:", len(expected_slice))
print("Expected first:", expected_slice["open_time_utc"].iloc[0])
print("Expected last:", expected_slice["open_time_utc"].iloc[-1])
print("PASS: Deliverable 1 accepts both naive and UTC-aware fromdate/todate, with identical inclusive slicing.")


,case,from_arg_type,from_arg_tzinfo,to_arg_tzinfo,count,first_dt,last_dt
0,naive datetime interpreted as UTC,datetime,None,None,72,2024-01-01,2024-01-03 23:00:00
1,UTC-aware datetime,datetime,UTC,UTC,72,2024-01-01,2024-01-03 23:00:00


Expected count: 72
Expected first: 2024-01-01 00:00:00+00:00
Expected last: 2024-01-03 23:00:00+00:00
PASS: Deliverable 1 accepts both naive and UTC-aware fromdate/todate, with identical inclusive slicing.


## Part B - Deliverable 2: execution model and slippage

This section uses deterministic toy data so the correct signal bar, fill bar, price, and commission can be inspected directly.

The execution model passes this section if:

- Buy and sell orders fill at the next bar open, not the signal bar close.
- Commission is exactly 7 bps of absolute trade value on each side.
- The broker is `AlphaBroker` and the cost model label is `effective_7bps_per_side`.
- A zero-volume fill bar defers execution to the next non-zero-volume bar.
- More than 24 consecutive zero-volume fill attempts cancel the order.

### B1. Toy data and strategy helpers

In [6]:
tmpdir_ctx = tempfile.TemporaryDirectory()
TMPDIR = Path(tmpdir_ctx.name)
print("Temporary validation parquet directory:", TMPDIR)


def make_toy_df(n_bars=8, volumes=None):
    if volumes is None:
        volumes = [10.0] * n_bars
    assert len(volumes) == n_bars
    return pd.DataFrame({
        "open_time_utc": pd.date_range("2024-01-01 00:00:00", periods=n_bars, freq="h", tz="UTC").astype("datetime64[ms, UTC]"),
        "open": [100.0 + i for i in range(n_bars)],
        "high": [101.0 + i for i in range(n_bars)],
        "low": [99.0 + i for i in range(n_bars)],
        "close": [100.0 + i for i in range(n_bars)],
        "volume": volumes,
        "quote_volume": [1000.0] * n_bars,
        "trade_count": [10] * n_bars,
        "source": pd.array(["binance_vision"] * n_bars, dtype="string"),
        "ingested_at_utc": pd.Timestamp("2026-01-01 00:00:00", tz="UTC").floor("ms"),
    })


def write_toy_parquet(df, name):
    path = TMPDIR / name
    out = df.copy()
    out["ingested_at_utc"] = out["ingested_at_utc"].astype("datetime64[ms, UTC]")
    out.to_parquet(path, engine="pyarrow", index=False)
    return path


def expected_commission(size, price):
    return abs(size) * price * COMMISSION_RATE


class BuySellOnBars(bt.Strategy):
    params = (
        ("buy_bar", 3),
        ("sell_bar", 5),
    )

    def __init__(self):
        self.bar_log = []
        self.signals = []
        self.fills = []
        self.cancelled = []

    def next(self):
        bar_num = len(self)  # 1-based Backtrader bar number
        dt = self.data.datetime.datetime(0)
        self.bar_log.append({
            "bar_num": bar_num,
            "dt": dt,
            "open": float(self.data.open[0]),
            "close": float(self.data.close[0]),
            "volume": float(self.data.volume[0]),
            "position": float(self.position.size),
        })

        if bar_num == self.p.buy_bar and not self.position:
            self.signals.append({"side": "buy", "bar_num": bar_num, "dt": dt, "close": float(self.data.close[0])})
            self.buy()
        elif bar_num == self.p.sell_bar and self.position:
            self.signals.append({"side": "sell", "bar_num": bar_num, "dt": dt, "close": float(self.data.close[0])})
            self.sell()

    def notify_order(self, order):
        if order.status == order.Completed:
            self.fills.append({
                "status": order.getstatusname(),
                "side": "buy" if order.isbuy() else "sell",
                "executed_dt": bt.num2date(order.executed.dt).replace(tzinfo=None),
                "price": float(order.executed.price),
                "size": float(order.executed.size),
                "value": float(order.executed.value),
                "comm": float(order.executed.comm),
            })
        elif order.status == order.Canceled:
            self.cancelled.append({
                "status": order.getstatusname(),
                "side": "buy" if order.isbuy() else "sell",
                "ref": order.ref,
            })


def run_execution_check(path, strategy=BuySellOnBars, **strategy_kwargs):
    feed = ParquetFeed.from_parquet(path)
    cerebro = bt.Cerebro()
    cost_model = configure_cerebro(cerebro)
    assert isinstance(cerebro.broker, AlphaBroker)
    assert cost_model.fee_model_label == "effective_7bps_per_side"
    assert cost_model.effective_commission == COMMISSION_RATE
    cerebro.adddata(feed)
    cerebro.addstrategy(strategy, **strategy_kwargs)
    strat = cerebro.run()[0]
    return strat, cost_model

toy = make_toy_df()
toy_path = write_toy_parquet(toy, "toy_validation.parquet")
display(toy[["open_time_utc", "open", "high", "low", "close", "volume"]])


Temporary validation parquet directory: /var/folders/rq/573stz453md6gpcyqpz5y_dm0000gn/T/tmpoji3zc38


,open_time_utc,open,high,low,close,volume
0,2024-01-01 00:00:00+00:00,100.0,101.0,99.0,100.0,10.0
1,2024-01-01 01:00:00+00:00,101.0,102.0,100.0,101.0,10.0
2,2024-01-01 02:00:00+00:00,102.0,103.0,101.0,102.0,10.0
3,2024-01-01 03:00:00+00:00,103.0,104.0,102.0,103.0,10.0
4,2024-01-01 04:00:00+00:00,104.0,105.0,103.0,104.0,10.0
5,2024-01-01 05:00:00+00:00,105.0,106.0,104.0,105.0,10.0
6,2024-01-01 06:00:00+00:00,106.0,107.0,105.0,106.0,10.0
7,2024-01-01 07:00:00+00:00,107.0,108.0,106.0,107.0,10.0


### B2. Next-bar-open fill and 7 bps commission

In [7]:
strat, cost_model = run_execution_check(toy_path)

signals_df = pd.DataFrame(strat.signals)
fills_df = pd.DataFrame(strat.fills)
bar_log_df = pd.DataFrame(strat.bar_log)

display(signals_df)
display(fills_df)

assert len(strat.signals) == 2, f"Expected buy and sell signals, got {len(strat.signals)}"
assert len(strat.fills) == 2, f"Expected buy and sell fills, got {len(strat.fills)}"

buy_signal = strat.signals[0]
sell_signal = strat.signals[1]
buy_fill = strat.fills[0]
sell_fill = strat.fills[1]

expected_buy_dt = toy.loc[3, "open_time_utc"].tz_localize(None)
expected_sell_dt = toy.loc[5, "open_time_utc"].tz_localize(None)
expected_buy_open = toy.loc[3, "open"]
expected_sell_open = toy.loc[5, "open"]
expected_buy_comm = expected_commission(buy_fill["size"], buy_fill["price"])
expected_sell_comm = expected_commission(sell_fill["size"], sell_fill["price"])

assert buy_signal["side"] == "buy"
assert buy_signal["bar_num"] == 3
assert buy_fill["side"] == "buy"
assert buy_fill["executed_dt"] == expected_buy_dt
assert buy_fill["price"] == expected_buy_open
assert buy_fill["price"] != buy_signal["close"], "Buy filled at signal close instead of next bar open"
assert buy_fill["comm"] == expected_buy_comm

assert sell_signal["side"] == "sell"
assert sell_signal["bar_num"] == 5
assert sell_fill["side"] == "sell"
assert sell_fill["executed_dt"] == expected_sell_dt
assert sell_fill["price"] == expected_sell_open
assert sell_fill["price"] != sell_signal["close"], "Sell filled at signal close instead of next bar open"
assert sell_fill["comm"] == expected_sell_comm

b2_checks = pd.DataFrame([
    {"check": "buy fill datetime", "expected": expected_buy_dt, "actual": buy_fill["executed_dt"], "pass": buy_fill["executed_dt"] == expected_buy_dt},
    {"check": "buy fill open", "expected": expected_buy_open, "actual": buy_fill["price"], "pass": buy_fill["price"] == expected_buy_open},
    {"check": "buy commission", "expected": expected_buy_comm, "actual": buy_fill["comm"], "pass": buy_fill["comm"] == expected_buy_comm},
    {"check": "sell fill datetime", "expected": expected_sell_dt, "actual": sell_fill["executed_dt"], "pass": sell_fill["executed_dt"] == expected_sell_dt},
    {"check": "sell fill open", "expected": expected_sell_open, "actual": sell_fill["price"], "pass": sell_fill["price"] == expected_sell_open},
    {"check": "sell commission", "expected": expected_sell_comm, "actual": sell_fill["comm"], "pass": sell_fill["comm"] == expected_sell_comm},
])
display(b2_checks)

print(f"Expected buy fill open: {expected_buy_open}; actual buy fill price: {buy_fill['price']}")
print(f"Expected buy commission: {expected_buy_comm}; actual buy commission: {buy_fill['comm']}")
print(f"Expected sell fill open: {expected_sell_open}; actual sell fill price: {sell_fill['price']}")
print(f"Expected sell commission: {expected_sell_comm}; actual sell commission: {sell_fill['comm']}")
print("PASS: Deliverable 2 next-bar-open execution and 7bps commission validated.")


,side,bar_num,dt,close
0,buy,3,2024-01-01 02:00:00,102.0
1,sell,5,2024-01-01 04:00:00,104.0


,status,side,executed_dt,price,size,value,comm
0,Completed,buy,2024-01-01 03:00:00,103.0,1.0,103.0,0.0721
1,Completed,sell,2024-01-01 05:00:00,105.0,-1.0,103.0,0.0735


,check,expected,actual,pass
0,buy fill datetime,2024-01-01 03:00:00,2024-01-01 03:00:00,True
1,buy fill open,103.0,103.0,True
2,buy commission,0.0721,0.0721,True
3,sell fill datetime,2024-01-01 05:00:00,2024-01-01 05:00:00,True
4,sell fill open,105.0,105.0,True
5,sell commission,0.0735,0.0735,True


Expected buy fill open: 103.0; actual buy fill price: 103.0
Expected buy commission: 0.0721; actual buy commission: 0.0721
Expected sell fill open: 105.0; actual sell fill price: 105.0
Expected sell commission: 0.0735; actual sell commission: 0.0735
PASS: Deliverable 2 next-bar-open execution and 7bps commission validated.


### B3. Zero-volume fill defers to the next valid bar

In [8]:
toy_zv = toy.copy()
toy_zv.loc[3, "volume"] = 0.0  # The normal buy fill bar is zero-volume.
toy_zv_path = write_toy_parquet(toy_zv, "toy_zero_volume.parquet")

display(toy_zv[["open_time_utc", "open", "close", "volume"]])

zv_strat, _ = run_execution_check(toy_zv_path)
zv_signals_df = pd.DataFrame(zv_strat.signals)
zv_fills_df = pd.DataFrame(zv_strat.fills)

display(zv_signals_df)
display(zv_fills_df)

assert len(zv_strat.fills) == 2, f"Expected deferred buy plus later sell, got {len(zv_strat.fills)} fills"
zv_buy_fill = zv_strat.fills[0]

signal_bar_num = 3
intended_fill_bar_num = 4
actual_fill_bar_num = 5
intended_fill_idx = intended_fill_bar_num - 1
actual_fill_idx = actual_fill_bar_num - 1

assert zv_buy_fill["side"] == "buy"
assert toy_zv.loc[intended_fill_idx, "volume"] == 0.0
assert zv_buy_fill["executed_dt"] == toy_zv.loc[actual_fill_idx, "open_time_utc"].tz_localize(None)
assert zv_buy_fill["price"] == toy_zv.loc[actual_fill_idx, "open"]
assert zv_buy_fill["comm"] == expected_commission(zv_buy_fill["size"], zv_buy_fill["price"])

print(f"Signal at bar {signal_bar_num}: dt={toy_zv.loc[signal_bar_num - 1, 'open_time_utc']}, close={toy_zv.loc[signal_bar_num - 1, 'close']}")
print(f"Intended fill at bar {intended_fill_bar_num} open={toy_zv.loc[intended_fill_idx, 'open']}, volume={toy_zv.loc[intended_fill_idx, 'volume']}")
print(f"Actual fill deferred to bar {actual_fill_bar_num} open={toy_zv.loc[actual_fill_idx, 'open']}, volume={toy_zv.loc[actual_fill_idx, 'volume']}")
print(f"Deferred buy actual price: {zv_buy_fill['price']}; expected: {toy_zv.loc[actual_fill_idx, 'open']}")
print("PASS: Deliverable 2 zero-volume deferral validated.")


,open_time_utc,open,close,volume
0,2024-01-01 00:00:00+00:00,100.0,100.0,10.0
1,2024-01-01 01:00:00+00:00,101.0,101.0,10.0
2,2024-01-01 02:00:00+00:00,102.0,102.0,10.0
3,2024-01-01 03:00:00+00:00,103.0,103.0,0.0
4,2024-01-01 04:00:00+00:00,104.0,104.0,10.0
5,2024-01-01 05:00:00+00:00,105.0,105.0,10.0
6,2024-01-01 06:00:00+00:00,106.0,106.0,10.0
7,2024-01-01 07:00:00+00:00,107.0,107.0,10.0


,side,bar_num,dt,close
0,buy,3,2024-01-01 02:00:00,102.0
1,sell,5,2024-01-01 04:00:00,104.0


,status,side,executed_dt,price,size,value,comm
0,Completed,buy,2024-01-01 04:00:00,104.0,1.0,104.0,0.0728
1,Completed,sell,2024-01-01 05:00:00,105.0,-1.0,104.0,0.0735


Signal at bar 3: dt=2024-01-01 02:00:00+00:00, close=102.0
Intended fill at bar 4 open=103.0, volume=0.0
Actual fill deferred to bar 5 open=104.0, volume=10.0
Deferred buy actual price: 104.0; expected: 104.0
PASS: Deliverable 2 zero-volume deferral validated.


### B4. More than 24 zero-volume bars cancels the order

In [9]:
class BuyAndHoldOnBar(bt.Strategy):
    params = (("buy_bar", 3),)

    def __init__(self):
        self.signals = []
        self.fills = []
        self.cancelled = []
        self.order_events = []
        self.final_defer_counts = None
        self.final_pending_live_refs = None
        self.final_submitted_live_refs = None

    def next(self):
        bar_num = len(self)
        dt = self.data.datetime.datetime(0)
        if bar_num == self.p.buy_bar and not self.position:
            order = self.buy()
            self.signals.append({
                "event": "signal_submitted",
                "side": "buy",
                "bar_num": bar_num,
                "dt": dt,
                "close": float(self.data.close[0]),
                "order_ref": order.ref,
            })

    def notify_order(self, order):
        dt = self.data.datetime.datetime(0) if len(self) else None
        event = {
            "event": "notify_order",
            "status": order.getstatusname(),
            "status_code": int(order.status),
            "side": "buy" if order.isbuy() else "sell",
            "ref": order.ref,
            "bar_num_seen_by_strategy": len(self),
            "dt_seen_by_strategy": dt,
            "alive_after_notify": bool(order.alive()),
        }
        if order.status == order.Completed:
            event.update({
                "executed_dt": bt.num2date(order.executed.dt).replace(tzinfo=None),
                "price": float(order.executed.price),
                "size": float(order.executed.size),
                "comm": float(order.executed.comm),
            })
            self.fills.append(event.copy())
        elif order.status == order.Canceled:
            self.cancelled.append(event.copy())
        self.order_events.append(event)

    def stop(self):
        broker = self.broker
        self.final_defer_counts = dict(getattr(broker, "_defer_counts", {}))
        self.final_pending_live_refs = [
            order.ref
            for order in list(getattr(broker, "pending", []))
            if hasattr(order, "alive") and order.alive()
        ]
        self.final_submitted_live_refs = [
            order.ref
            for order in list(getattr(broker, "submitted", []))
            if hasattr(order, "alive") and order.alive()
        ]

long_volumes = [10.0] * 40
for i in range(3, 28):  # 25 zero-volume bars after the buy signal, exceeding MAX_DEFER_BARS=24.
    long_volumes[i] = 0.0

toy_cancel = make_toy_df(n_bars=40, volumes=long_volumes)
toy_cancel_path = write_toy_parquet(toy_cancel, "toy_zero_volume_cancel.parquet")

cancel_strat, _ = run_execution_check(toy_cancel_path, strategy=BuyAndHoldOnBar, buy_bar=3)

signals_cancel_df = pd.DataFrame(cancel_strat.signals)
events_cancel_df = pd.DataFrame(cancel_strat.order_events)
fills_cancel_df = pd.DataFrame(cancel_strat.fills)
cancelled_df = pd.DataFrame(cancel_strat.cancelled)
zero_window = toy_cancel.loc[3:27, ["open_time_utc", "open", "volume"]].copy()
zero_window["bar_num"] = zero_window.index + 1

display(zero_window.head(3))
display(zero_window.tail(3))
display(signals_cancel_df)
display(events_cancel_df)
display(fills_cancel_df)
display(cancelled_df)
print("Final broker defer counts:", cancel_strat.final_defer_counts)
print("Final live pending refs:", cancel_strat.final_pending_live_refs)
print("Final live submitted refs:", cancel_strat.final_submitted_live_refs)

assert len(signals_cancel_df) == 1
assert len(zero_window) == 25
assert (zero_window["volume"] == 0).all()
assert len(cancel_strat.fills) == 0, f"Order should cancel instead of filling, got fills: {cancel_strat.fills}"
assert len(cancel_strat.cancelled) == 1, f"Expected one cancellation, got: {cancel_strat.cancelled}"
assert cancel_strat.cancelled[0]["status"] == "Canceled"
assert cancel_strat.cancelled[0]["alive_after_notify"] is False

statuses = events_cancel_df["status"].tolist()
assert "Submitted" in statuses, statuses
assert "Accepted" in statuses, statuses
assert "Canceled" in statuses, statuses
assert statuses.index("Submitted") < statuses.index("Accepted") < statuses.index("Canceled"), statuses
assert statuses[-1] == "Canceled", statuses

assert cancel_strat.final_defer_counts == {}, cancel_strat.final_defer_counts
assert cancel_strat.final_pending_live_refs == [], cancel_strat.final_pending_live_refs
assert cancel_strat.final_submitted_live_refs == [], cancel_strat.final_submitted_live_refs

print("PASS: Deliverable 2 cancel path validated: 25 zero-volume bars -> Canceled, notify order is sane, no ghost order remains.")


Order 5 cancelled: deferred 25 bars on zero-volume


,open_time_utc,open,volume,bar_num
3,2024-01-01 03:00:00+00:00,103.0,0.0,4
4,2024-01-01 04:00:00+00:00,104.0,0.0,5
5,2024-01-01 05:00:00+00:00,105.0,0.0,6


,open_time_utc,open,volume,bar_num
25,2024-01-02 01:00:00+00:00,125.0,0.0,26
26,2024-01-02 02:00:00+00:00,126.0,0.0,27
27,2024-01-02 03:00:00+00:00,127.0,0.0,28


,event,side,bar_num,dt,close,order_ref
0,signal_submitted,buy,3,2024-01-01 02:00:00,102.0,5


,event,status,status_code,side,ref,bar_num_seen_by_strategy,dt_seen_by_strategy,alive_after_notify
0,notify_order,Submitted,1,buy,5,4,2024-01-01 03:00:00,True
1,notify_order,Accepted,2,buy,5,4,2024-01-01 03:00:00,True
2,notify_order,Canceled,5,buy,5,28,2024-01-02 03:00:00,False


""


,event,status,status_code,side,ref,bar_num_seen_by_strategy,dt_seen_by_strategy,alive_after_notify
0,notify_order,Canceled,5,buy,5,28,2024-01-02 03:00:00,False


Final broker defer counts: {}
Final live pending refs: []
Final live submitted refs: []
PASS: Deliverable 2 cancel path validated: 25 zero-volume bars -> Canceled, notify order is sane, no ghost order remains.


## Final acceptance summary

In [10]:
print("Deliverable 1 PASS: ParquetFeed row count, OHLCV values, extra lines, timestamps, and date slicing match pandas.")
print("Deliverable 2 PASS: next-bar-open fills, 7bps commission, zero-volume deferral, and zero-volume cancellation are validated.")
print("Reminder: keep pytest as the primary correctness evidence; this notebook is the readable acceptance spot-check.")
print("Scope note: Deliverables 1/2 are accepted pending Phase 1A manual trade audit on real engine-generated trades after Deliverable 3.")


Deliverable 1 PASS: ParquetFeed row count, OHLCV values, extra lines, timestamps, and date slicing match pandas.
Deliverable 2 PASS: next-bar-open fills, 7bps commission, zero-volume deferral, and zero-volume cancellation are validated.
Reminder: keep pytest as the primary correctness evidence; this notebook is the readable acceptance spot-check.
Scope note: Deliverables 1/2 are accepted pending Phase 1A manual trade audit on real engine-generated trades after Deliverable 3.
